In [1]:
import pandas as pd

In [2]:
# Charger le fichier CSV dans un DataFrame
def load_csv_to_dataframe(csv_file_path):
    try:
        df = pd.read_csv(csv_file_path)
        return df
    except FileNotFoundError:
        print(f"Le fichier '{csv_file_path}' n'a pas été trouvé.")
        return None

In [3]:
# Extraire les listes de jeux à partir de la colonne cible
def extract_game_lists_from_dataframe(df, target_column_name):
    list_of_lists = df[target_column_name].dropna().apply(lambda x: [game.strip() for game in x.split(',')]).tolist()
    return list_of_lists

In [4]:
# Créer la matrice de cooccurrence à partir de la liste de listes
def create_cooccurrence_matrix(list_of_lists):
    # Créer une liste unique de tous les jeux
    all_games = set(game for games in list_of_lists for game in games)
    all_games = sorted(all_games)
    
    # Initialiser le tableau de cooccurrence
    cooccurrence_matrix = pd.DataFrame(0, index=all_games, columns=all_games)
    
    # Parcourir chaque liste de jeux et mettre à jour le tableau de cooccurrence
    for games in list_of_lists:
        for i in range(len(games)):
            for j in range(i, len(games)):
                game_i = games[i]
                game_j = games[j]
                cooccurrence_matrix.at[game_i, game_j] += 1
                if game_i != game_j:
                    cooccurrence_matrix.at[game_j, game_i] += 1
    
    return cooccurrence_matrix

In [5]:
# Calculer l'indicateur de présence conjointe pour chaque jeu
def calculate_joint_presence_indicator(cooccurrence_matrix):
    # Calculer la somme des cooccurrences pour chaque jeu
    joint_presence_indicator = cooccurrence_matrix.sum(axis=1)
    
    # Ajouter une colonne au DataFrame de la matrice de cooccurrence pour l'indicateur
    cooccurrence_matrix['Joint Presence Indicator'] = joint_presence_indicator
    
    return cooccurrence_matrix

In [10]:
# Nettoyer la matrice de cooccurrence
def clean_cooccurrence_matrix(cooccurrence_matrix):
    # Mettre à 0 toutes les cellules où i = j
    for game in cooccurrence_matrix.index:
        cooccurrence_matrix.at[game, game] = 0
    
    # Renommer les colonnes et les lignes
    #new_names = {name: f"Jeu_{i+1}" for i, name in enumerate(cooccurrence_matrix.index)}
    new_names = { '2XKO' : "2XKO", 'Among Us' : "Among_Us", 'Babyfoot' : "Babyfoot", 'Balatro' : "Balatro", 'Code Names' : "Codenames", 'GeoGuessr' : "Geoguessr", 'Golf it!' : "Golf_it", 'League of Legends' : "LoL", 'Loup-garou pour une nuit' : "LG_One_night", 'Mario Kart 8DX (Nintendo Switch)' : "MK8dx", 'Minecraft minigames' : "Minecraft_minigames", 'OSU' : "OSU", 'Overcooked 2' : "Overcooked2", 'Overwatch 2' : "Overwatch2", 'Peak speedrun' : "Peak", 'Ping-pong' : "Ping-pong", 'Refunct speedrun' : "Refunct", 'Rocket League (2 vs 2)' : "RL_2v2", 'Super Smash Bros Ultimate (1 vs 1) (Nintendo Switch)' : "Smash_1v1", 'Triviapoly' : "Triviapoly", 'Valorant' : "Valo", 'osu!' : "Osu", 'Échecs 960' : "Echecs960"}
    cooccurrence_matrix.rename(index=new_names, columns=new_names, inplace=True)
    
    return cooccurrence_matrix
    

In [31]:
# Exemple d'utilisation dans un Jupyter Notebook
csv_file_path = 'inscription_donnees.csv'
target_column_name = 'À quels tournois aimeriez-vous participer ?'

# Charger le DataFrame
df = load_csv_to_dataframe(csv_file_path)
if df is not None:
    # Extraire les listes de jeux
    liste_de_listes = extract_game_lists_from_dataframe(df, target_column_name)
    
    # Créer et afficher la matrice de cooccurrence
    cooccurrence_matrix = create_cooccurrence_matrix(liste_de_listes)

    #Calculer l'indicateur de présence conjointe
    #cooccurrence_matrix = calculate_joint_presence_indicator(cooccurrence_matrix)

    #Cleanup
    cooccurrence_matrix = clean_cooccurrence_matrix(cooccurrence_matrix)

    #Remove speedruns or other useless stats
    useless_games = ["Peak", "Refunct"]
    cooccurrence_matrix = cooccurrence_matrix.drop(useless_games)
    cooccurrence_matrix = cooccurrence_matrix.drop(useless_games, axis=1)
    
    display(cooccurrence_matrix)
    


,2XKO,Among_Us,Babyfoot,Balatro,Codenames,Geoguessr,Golf_it,LoL,LG_One_night,MK8dx,Minecraft_minigames,Overcooked2,Overwatch2,Ping-pong,RL_2v2,Smash_1v1,Triviapoly,Valo,Osu,Echecs960
2XKO,0,6,11,13,11,10,10,14,7,8,12,9,6,6,10,12,6,6,3,5
Among_Us,6,0,22,16,23,21,19,21,25,18,25,16,19,24,14,15,17,21,8,10
Babyfoot,11,22,0,19,20,20,17,21,20,17,22,20,14,23,15,19,19,16,4,7
Balatro,13,16,19,0,22,20,22,20,15,22,25,23,14,17,15,19,13,14,6,5
Codenames,11,23,20,22,0,22,21,21,23,24,30,26,14,22,13,21,19,16,7,11
Geoguessr,10,21,20,20,22,0,24,22,21,22,31,26,17,21,18,18,16,20,7,8
Golf_it,10,19,17,22,21,24,0,17,17,17,26,22,16,18,17,13,16,19,6,9
LoL,14,21,21,20,21,22,17,0,19,21,28,19,15,22,19,20,10,24,6,9
LG_One_night,7,25,20,15,23,21,17,19,0,18,21,19,14,21,11,20,14,17,8,8
MK8dx,8,18,17,22,24,22,17,21,18,0,28,22,12,20,13,23,13,15,6,5


In [26]:
#cooccurrence_matrix.to_excel('cooccurrence_matrix.xlsx', index=True)

In [27]:
cooccurrence_matrix.to_csv('cooccurrence_matrix_2025.csv', index=True)